### 대화내용을 저장하는 메모리
랭그래프에서 제공하는 '메모리'를 활용하여 대화내용을 간편하게 저장한다.

In [ ]:
# 1. 랭체인 설치
%pip install langchain

# 2. 랭체인에서 OpenAI를 사용하기 위한 패키지
%pip install langchain_openai

# 3. 랭그래프 설치
%pip install langgraph


# 2. 랭체인에서 OpenAI를 사용하기 위한 패키지
%pip install langchain_openai

# 3. 랭그래프 설치
%pip install langgraph


In [12]:

# import
import os
from langchain_openai import ChatOpenAI  # 랭체인에서 OpenAI 모델을 사용하기 위한 패키지
from langgraph.graph import StateGraph, START, END # 상태 그래프를 정의하기 위한 클래스와 상수 (전체 상태, 그래프 시작 노드, 종료 노드)
from langgraph.graph.message import add_messages # 상태 그래프에 메시지를 추가하기 위한 함수
from typing import Annotated # 타입 힌트를 위한 함수
from typing import TypedDict # State 클래스를 딕셔너리 형태로 관리하기 위해 사용
from dotenv import load_dotenv  # .env 파일에서 환경변수 불러오기 위한 패키지

load_dotenv()
model = ChatOpenAI(model="gpt-4o-mini")


### 일단 기본 챗봇을 만든다.
이전에 Langgraph 의 코드 사용

# *** add_messages 함수
add_messages:
상태의 messages 필드가 "덮어쓰기"가 아니라 "누적"되도록 만드는 함수다.
대화형 그래프에서는 거의 필수라고 생각하면 된다.
"
새 값이 들어오면 그냥 replace하지 말고
add_messages(left, right) 규칙으로 병합해라
"
라는 동작을 한다.

원래, 노드에서 대화 답변뿐만 아닌 이전 대화를 누적해서 return 하고 싶다면
return { "messages": state["messages"] + [response] } #원래 메세지 + 답변
의 형태로 반환해줘야 할 것이다.

하지만 state 내에서 
messages: Annotated[..., add_messages]
로 해놓으면 
노드 반환이
return {"messages": [response]}
형태로 답변만 반환하는 형태여도

State 의 "messages" 상태키의 인자는 add_messages 함수를 거쳐서 이후에는
[
    HumanMessage(content="안녕"),
    AIMessage(content="반가워요")
]
형태로 대화가 누적되는 형태로 받아준다.

In [23]:

# *** add messages : 대화 내용을 누적해주는 함수
from langgraph.graph.message import add_messages

# State 클래스
class State(TypedDict):
    messages: Annotated[list[str], add_messages] # *** add messages 데코레이터를 사용하여 대화 누적되는 상태로 저장

# StateGraph 객체 생성 (graph_builder)
graph_builder = StateGraph(State)

In [ ]:
def NODE_generate_response(state: State) -> str:
    messages = state["messages"] # 모델에 현재까지 누적된 대화 리스트
    response = model.invoke(messages) # 모델에 대화 리스트를 입력하여 응답 생성
    return{"messages": [response]} # AI 답변 1개만 State 에 반환하여 add_messages 누적 추가

graph_builder.add_node("generate_response", NODE_generate_response)
graph_builder.add_edge(START, "generate_response")
graph_builder.add_edge("generate_response", END)
MY_graph = graph_builder.compile()

### 기억하는 대화

# 시퀀스
MemorySaver()를 만들어서 - 그래프 상태를 메모리에 저장할 준비를 함.
graph_builder.compile(checkpointer=memory)로 체크포인터가 붙은 실행 가능한 그래프를 만듦.

thread_id = "ID_123"를 지정해서 - 이 대화가 어느 스레드인지 정함. 같은 ID면 이전 state를 이어받음.
사용자가 질문을 입력하면 그걸 HumanMessage로 감싸서 messages state로 그래프에 넣음.
MY_graph.stream(..., stream_mode="values")로 그래프 실행 중의 각 단계별 전체 state를 순서대로 받음.
각 단계마다 메시지 출력

# User Input 받는 시점
for event in MY_graph.stream(
        {"messages" : [HumanMessage(content=user_input)]},
        ...
)
그래프가 시작되기 전부터 messages 안에 이미 사용자 메시지가 들어 있음

# 각 Node 진행의 결과 State
event = 각 node 실행 후 반영된 state 결과 스냅샷
    for event in MY_graph.stream({"messages" : [HumanMessage(content=user_input)]}, ...):
        event["messages"][-1].pretty_print() 

# Memory Saver
RAM 메모리에 저장하는 체크 포인터 (종료시 휘발)
그래프 상태를 저장할 공간 확보
대화 기록 보관할 임시 창고
- 대화 기록
- 그래프 State

# checkpointer
그래프를 컴파일 할때 저장소를 연결한다.
이번에는 MemorySaver 를 연결하여
- 이 그래프는 실행될 때 마다 memory 를 사용해서 state 를 checkpoint 로 저장한다.
는 기능을 넣음

# config
그래프 설정
그래프 실행 시 사용할 설정값 딕셔너리

# thread_id
LangGraph는 checkpoint를 thread 단위로 관리


# stream_mode
stream_mode="values"  # "values": 각 step 후의 전체 state를 스트리밍
                       # "updates": 각 step에서 바뀐 부분만 스트리밍
                       # "messages": LLM이 생성하는 메시지/토큰 chunk를 스트리밍


In [25]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()
MY_graph = graph_builder.compile(checkpointer = memory)

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# *** Configurable : 대화 스레드 ID 설정
config_A= {"configurable" : {"thread_id" : "ID_123"}} # 대화 스레드 ID
config_B= {"configurable" : {"thread_id" : "ID_user99"}} # 대화 스레드 ID

while True:
    # 1. 대화 스레드 선택
    thread_select = input("ID 선택 (A or B): ")
    if thread_select.lower() == "a":
        config_now = config_A
    elif thread_select.lower() == "b":
        config_now = config_B
    else:
        print("대화를 종료합니다.")
        break

    room = config_now["configurable"]["thread_id"]

    # 중단조건설정
    user_input = input(f"[{room}]YOU : ")
    if user_input.lower() in ["quit", "q", "ㅂ"]:
        print("대화를 종료합니다.")
        break

    # User Massage
    for event in MY_graph.stream(
        {"messages" : [HumanMessage(content=user_input)]}, # 사용자 입력을 HumanMessage 객체로 래핑하여 State 의 messages 리스트에 '직접' 추가
        config = config_now,       # 대화 스레드 ID 설정
        stream_mode="values" # 메세지 스트리밍
    ):
        event["messages"][-1].pretty_print() # -1:마지막메세지==>모델이생성한응답메시지
        
        print(f"현재 메세지 개수 : {len(event['messages'])}")

================================ Human Message =================================

안녕 나는 김철수
현재 메세지 개수 : 1
================================== Ai Message ==================================

안녕하세요, 김철수님! 어떻게 도와드릴까요?
현재 메세지 개수 : 2
================================ Human Message =================================

내 이름이 뭐야?
현재 메세지 개수 : 3
================================== Ai Message ==================================

당신의 이름은 김철수입니다. 다른 질문이나 필요한 정보가 있으면 말씀해 주세요!
현재 메세지 개수 : 4
================================ Human Message =================================

너는 내이름을 아니?
현재 메세지 개수 : 1
================================== Ai Message ==================================

죄송하지만, 저는 사용자의 이름이나 개인 정보를 알 수 없습니다. 어떻게 도와드릴까요?
현재 메세지 개수 : 2
================================ Human Message =================================

너는 내이름을 아니?
현재 메세지 개수 : 5
================================== Ai Message ==================================

네, 당신의 이름은 김철수라고 말씀하셨습니다. 궁금한 점이 있으면 언제든지 질문해 주세요!
현재 메세지 개수 : 6
==================